# Consolidate Radar Events

Merges detector fragments that are consistent in time, range, and range rate into unique radar events for catalog association and population inference.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = PROJECT_ROOT
CAT = BASE / 'phase0_outputs/catalog_matching/ca_cfar_1p75_24h'
OUT = BASE / 'phase0_outputs/catalog_matching/ca_cfar_1p75_24h_consolidated'
OUT.mkdir(parents=True, exist_ok=True)

SUMMARY = CAT / 'catalog_association_summary.csv'
CANDIDATES = CAT / 'catalog_candidates_top25.csv.gz'
assert SUMMARY.exists(), f'Missing {SUMMARY}'
assert CANDIDATES.exists(), f'Missing {CANDIDATES}'

                                       
MAX_GAP_SEC = 7.0
MAX_EXTRAPOLATED_RANGE_DISAGREEMENT_KM = 0.75
MAX_RATE_DISAGREEMENT_KM_S = 0.15
MIN_DURATION_SEC = 0.8

events = pd.read_csv(SUMMARY)
candidates = pd.read_csv(CANDIDATES)
print('Loaded detector fragments:', len(events))
print('Loaded saved catalog-candidate rows:', len(candidates))
print('Output:', OUT)


In [ ]:
def first_column(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    raise KeyError(f'None of {names} found. Available: {list(frame.columns)}')

id_col = first_column(events, ['object_detection_id', 'detection_id', 'track_id'])
time_col = first_column(events, ['reference_time_unix', 'fit_center_time_unix', 'reference_time'])
range_col = first_column(events, ['observed_range_km', 'reference_range_km', 'range_km'])
rate_col = first_column(events, ['observed_range_rate_km_s', 'reference_range_rate_km_s', 'range_rate_km_s'])
duration_col = first_column(events, ['duration_sec', 'duration_s'])

events = events.copy()
events['_id'] = events[id_col].astype(str)
events['_time'] = pd.to_numeric(events[time_col], errors='coerce')
if events['_time'].median() < 1e9:
    events['_time'] = pd.to_datetime(events[time_col], utc=True).astype('int64') / 1e9
events['_range'] = pd.to_numeric(events[range_col], errors='coerce')
events['_rate'] = pd.to_numeric(events[rate_col], errors='coerce')
events['_duration'] = pd.to_numeric(events[duration_col], errors='coerce')
events = events.sort_values('_time').reset_index(drop=True)
assert events[['_time','_range','_rate']].notna().all().all()

class UF:
    def __init__(self,n): self.p=list(range(n))
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a!=b: self.p[b]=a

uf=UF(len(events)); links=[]
for i in range(len(events)):
    ti,ri,vi,di = events.loc[i, ['_time','_range','_rate','_duration']]
    end_i = ti + di/2
    for j in range(i+1,len(events)):
        tj,rj,vj,dj = events.loc[j, ['_time','_range','_rate','_duration']]
        gap = (tj-dj/2)-end_i
        if gap > MAX_GAP_SEC: break
        if gap < -0.4: continue
        mid=(ti+tj)/2
        ri_mid=ri+vi*(mid-ti); rj_mid=rj+vj*(mid-tj)
        dr=abs(ri_mid-rj_mid); dv=abs(vi-vj)
        if dr <= MAX_EXTRAPOLATED_RANGE_DISAGREEMENT_KM and dv <= MAX_RATE_DISAGREEMENT_KM_S:
            uf.union(i,j); links.append((i,j,gap,dr,dv))

groups={}
for i in range(len(events)): groups.setdefault(uf.find(i),[]).append(i)
print('Accepted fragment links:',len(links))
print('Consolidated radar events:',len(groups),'from',len(events),'fragments')
print('Multi-fragment events:',sum(len(v)>1 for v in groups.values()))


In [ ]:
rows=[]; membership=[]
for number, inds in enumerate(sorted(groups.values(), key=lambda z: events.loc[z,'_time'].min()),1):
    g=events.loc[inds].sort_values('_time')
    t=g['_time'].to_numpy(float); r=g['_range'].to_numpy(float)
    center=float(np.median(t)); x=t-center
    if len(g)>=2 and np.ptp(t)>0:
        coef=np.polyfit(x,r,1); rate=float(coef[0]); center_range=float(np.polyval(coef,0)); resid=r-np.polyval(coef,x)
        rmse=float(np.sqrt(np.mean(resid**2)))
    else:
        rate=float(g['_rate'].iloc[0]); center_range=float(g['_range'].iloc[0]); rmse=np.nan
    start=float(np.min(t-g['_duration'].to_numpy(float)/2)); end=float(np.max(t+g['_duration'].to_numpy(float)/2))
    cid=f'C{number:04d}'
    rows.append(dict(consolidated_detection_id=cid, fit_center_time_unix=center,
        fit_center_time_utc=pd.to_datetime(center,unit='s',utc=True).isoformat(),
        fitted_range_km=center_range, fitted_range_rate_km_s=rate,
        start_time_unix=start,end_time_unix=end,duration_sec=end-start,
        trajectory_rmse_km=rmse,number_fragments=len(g),
        member_detection_ids=';'.join(g['_id'])))
    membership += [dict(consolidated_detection_id=cid,object_detection_id=x) for x in g['_id']]

consolidated=pd.DataFrame(rows)
membership=pd.DataFrame(membership)
consolidated.to_csv(OUT/'consolidated_object_detections.csv',index=False)
membership.to_csv(OUT/'fragment_membership.csv',index=False)
pd.DataFrame(links,columns=['first_index','second_index','gap_sec','range_disagreement_km','rate_disagreement_km_s']).to_csv(OUT/'accepted_fragment_links.csv',index=False)
display(consolidated.head(10))


In [ ]:
cand_id_col=first_column(candidates,['object_detection_id','detection_id','track_id'])
norad_col=first_column(candidates,['NORAD_CAT_ID','norad_cat_id','catalog_id','candidate_NORAD_CAT_ID'])
range_res_col=first_column(candidates,['abs_range_residual_km','range_residual_abs_km','range_residual_km'])
rate_res_col=first_column(candidates,['abs_range_rate_residual_km_s','range_rate_residual_abs_km_s','range_rate_residual_km_s'])

candidates=candidates.copy(); candidates['_id']=candidates[cand_id_col].astype(str)
candidates['_dr']=pd.to_numeric(candidates[range_res_col],errors='coerce').abs()
candidates['_dv']=pd.to_numeric(candidates[rate_res_col],errors='coerce').abs()
if candidates['_dv'].median()>1: candidates['_dv']/=1000.0

assoc=[]
for row in consolidated.itertuples(index=False):
    mids=set(row.member_detection_ids.split(';'))
    q=candidates[candidates['_id'].isin(mids)].copy()
                                                   
    range_gate=max(1.0,0.001*row.fitted_range_km)
    rate_gate=max(0.020,0.01*abs(row.fitted_range_rate_km_s))
    q=q[(q['_dr']<=range_gate)&(q['_dv']<=rate_gate)]
    if len(q):
        q['_norm']=np.hypot(q['_dr']/range_gate,q['_dv']/rate_gate)
        ranked=q.groupby(norad_col)['_norm'].median().sort_values()
        best=ranked.index[0]; status='unique' if len(ranked)==1 or ranked.iloc[1]-ranked.iloc[0]>0.15 else 'ambiguous'
        score=float(ranked.iloc[0]); n_candidates=int(len(ranked))
    else: best=np.nan; status='none'; score=np.nan; n_candidates=0
    assoc.append(dict(consolidated_detection_id=row.consolidated_detection_id,
        association_status=status,matched_NORAD_CAT_ID=best,
        normalized_residual_score=score,passing_catalog_candidates=n_candidates))

result=consolidated.merge(pd.DataFrame(assoc),on='consolidated_detection_id')
result.to_csv(OUT/'consolidated_catalog_association_summary.csv',index=False)
counts=result.association_status.value_counts().reindex(['unique','ambiguous','none'],fill_value=0)
summary={'input_fragments':int(len(events)),'consolidated_events':int(len(result)),
         'unique_associations':int(counts['unique']),'ambiguous_associations':int(counts['ambiguous']),
         'unassociated_events':int(counts['none']),'unique_catalog_objects':int(result.loc[result.association_status=='unique','matched_NORAD_CAT_ID'].nunique()),
         'settings':{'max_gap_sec':MAX_GAP_SEC,'max_range_disagreement_km':MAX_EXTRAPOLATED_RANGE_DISAGREEMENT_KM,'max_rate_disagreement_km_s':MAX_RATE_DISAGREEMENT_KM_S}}
(OUT/'consolidation_campaign_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

fig,ax=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
ax[0].bar(['Original fragments','Consolidated events'],[len(events),len(result)],color=['#B7C4CF','#176B87'])
ax[0].set_ylabel('Count'); ax[0].set_title('(a) Fragment consolidation',loc='left',fontweight='bold')
ax[1].bar(['Associated','Ambiguous','Unassociated'],[counts['unique'],counts['ambiguous'],counts['none']],color=['#176B87','#F2A541','#B7C4CF'])
ax[1].set_ylabel('Consolidated events'); ax[1].set_title('(b) Catalog status after consolidation',loc='left',fontweight='bold')
for a in ax:
    for p in a.patches: a.text(p.get_x()+p.get_width()/2,p.get_height(),str(int(p.get_height())),ha='center',va='bottom',fontweight='bold')
fig.savefig(OUT/'consolidation_summary.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT/'consolidation_summary.pdf',bbox_inches='tight')
plt.show()
print('FINISHED. Outputs saved to:',OUT)


## Successful finish
